In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Hybrid ML

In [ ]:
#compute climatology from assimilation
path = '/work/uo1075/u241308/ML_infilling/hfls/'
file ='asseikeraf_hfls_r1-16i8p4_360x180.nc'

ref_min = 1985
ref_max = 2014

with xr.open_dataset(path+file,decode_times=False) as hfls_clim:
    hfls_clim = hfls_clim.ahfl.mean(dim='sfc') #average over ensemble members
    #Time index is not readable for .dt. We create a new index
    reference_date = str('1958-1-01')
    print("Reference starting date: " + reference_date)
    hfls_clim['time'] = pd.date_range(start=reference_date, periods=hfls_clim.sizes['time'], freq='MS')
    
    hfls_clim = hfls_clim[(hfls_clim.time.dt.year>=ref_min)&(hfls_clim.time.dt.year<=ref_max)]
    hfls_clim = hfls_clim.groupby('time.month').mean(dim='time')

In [ ]:
path_in = '/work/uo1075/u241308/mpiesm-1.2.01p7-levante/proccessed_output/hfls/'
path_out = '/work/uo1075/u241308/data_python_PostDoc/ML_assimilation/hfls_ML_NA/anomaly/'
y_start = 1958
y_end = 2021

for y_id in range(y_start,y_end):
    file_in = 'hfls_%i_r1-16i2p3-LR_26_months_360x180.nc' %y_id
    file_out = 'hfls_%i_r1-16i2p3-LR_26_months_360x180_anomaly_ref_%i-%i.nc' %(y_id,ref_min,ref_max)
    with xr.open_dataset(path_in+file_in) as hfls:
        hfls = hfls.hfls
    hfls_anomaly = hfls.groupby('time.month') - hfls_clim
    hfls_anomaly.to_netcdf(path_out+file_out)
    print('Year %i done'%y_id)

# Standard

In [ ]:
path_in = '/work/uo1075/u241308/data_python_PostDoc/ML_assimilation/hfls_benchmark/processed/'
path_out = '/work/uo1075/u241308/data_python_PostDoc/ML_assimilation/hfls_benchmark/anomaly/'
y_start = 1960
y_end = 2020

for y_id in range(y_start,y_end):
    file_in = 'hfls_%i_r1-16i2p2-LR_3_years_360x180.nc' %y_id
    file_out = 'hfls_%i_r1-16i2p2-LR_3_years_360x180_anomaly_ref_%i-%i.nc' %(y_id,ref_min,ref_max)
    with xr.open_dataset(path_in+file_in,decode_times=False) as hfls:
        hfls = hfls.hfls
        #time axis is wrong. we set a new one
        units, reference_date = hfls.time.attrs['units'].split('since')
        list1 = list(reference_date)
        list1[9:11]='01'
        reference_date = ''.join(list1)
        hfls['time'] = pd.date_range(start=reference_date, periods=hfls.sizes['time'], freq='MS')
    hfls_anomaly = hfls.groupby('time.month') - hfls_clim
    hfls_anomaly.to_netcdf(path_out+file_out)
    print('Year %i done'%y_id)

# ERA5

In [25]:
path = '/work/uo1075/u241308/data_python_PostDoc/ML_assimilation/era5/'
file = 'era5_hfls_1940-2024_360x180.nc'
file_out = 'era5_hfls_1940-2024_360x180_anomaly_ref_%i-%i.nc' %(ref_min,ref_max)

era = xr.open_dataset(path+file).slhf
#change 'valid_time' dimension name to 'time'
era = era.rename({"valid_time": "time"})

##convert J/m2 to W/m2, dividing by each month's seconds
#time_index = pd.to_datetime(era.time.values)
#seconds_per_month = np.array([t.days_in_month * 24 * 3600 for t in time_index])
#era = era / xr.DataArray(seconds_per_month, coords=[era.time], dims=['time'])

#convert J/m2 to W/m2, dividing by 86400
era = era/86400 

#compute climatology and anomalies
era_clim = era[(era.time.dt.year>=ref_min)&(era.time.dt.year<=ref_max)].groupby('time.month').mean(dim='time')
era_anomaly = era.groupby('time.month') - era_clim

#save
era_anomaly.name = 'slhf'
era_anomaly.to_netcdf(path+file_out)